<a href="https://colab.research.google.com/github/janpeter19/BPL_TEST2_Perfusion/blob/main/BPL_TEST2_Perfusion_oms_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BPL_TEST2_Perfusion script with OMSimulator

The key library OMSimulator is installed.

After the installation a small application BPL_TEST2_Perfusion is loaded and run. You can continue with this example if you like.

In [1]:
!lsb_release -a # Actual VM Ubuntu version used by Google

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 24.04.5 LTS
Release:	24.04
Codename:	noble


In [2]:
!python --version

Python 3.13.18


In [3]:
%pip install -q uv

In [4]:
!uv pip install OMSimulator

Using Python 3.13.18 environment at: /usr
Resolved 7 packages in 396ms
Prepared 1 package in 5.93s
Installed 1 package in 3ms
 + omsimulator==3.0.0.post198


In [5]:
!uv pip install --system git+https://github.com/janpeter19/FMU_explore.git@v1.2.6

Using Python 3.13.18 environment at: /usr
Resolved 1 package in 496ms
Prepared 1 package in 4.59s
Installed 1 package in 1ms
 + fmu-explore==1.2.6 (from git+https://github.com/janpeter19/FMU_explore.git@573ae8b5a2daf5d983a5cf2d98abbd78ab7da18e)


In [6]:
!uv pip install DyMat

Using Python 3.13.18 environment at: /usr
Resolved 1 package in 1.08s
Prepared 1 package in 336ms
Installed 1 package in 1ms
 + dymat==0.7


# Notes of BPL_TEST2_Perfusion

This notebook explore perfusion cultivation in comparison with ordinary continuous cultivation (chemostat) and use comparable settings to earlier notebook. Further you see here examples of interaction with the simplified commands par(), init(), simu() etc as well as direct interaction with the FMU which is called "model" here. The last simulation is always available in the workspace and called "sim_res". Note that describe() brings mainly up from descriptive information from the Modelica code from the FMU but is complemented by some information given in the Python setup file.

Now specific installation run a simulation and notebook for that
Start with connecting to Github. Then upload the two files:

* FMU - BPL_TEST2_Perfusion_linux_om_me.fmu
* Setup-file - BPL_TEST2_Perfusion_oms.py
* Setup-file - BPL_TEST2_Perfusion_oms_explore.py

In [10]:
%%bash
git clone https://github.com/janpeter19/BPL_TEST2_Perfusion

Cloning into 'BPL_TEST2_Perfusion'...


In [11]:
%cd BPL_TEST2_Perfusion

/content/BPL_TEST2_Perfusion/BPL_TEST2_Perfusion


In [12]:
# Import the application model and context variables
from BPL_TEST2_Perfusion_oms import*

IndentationError: expected an indented block after 'if' statement on line 41 (BPL_TEST2_Perfusion_oms.py, line 44)

In [ ]:
# Import the general FMU_explore functions ana adapt them to the application
from fmu_explore_oms import AdaptWith

Application = AdaptWith(parValue, parLocation, parCheck, \
                        fmu_model, fmu_process_diagram, \
                        MSL_usage, MSL_version, BPL_version, \
                        options, simulationTime, timeDiscreteStates, \
                        diagrams, ax, lines, external_function = cstrProdMax)

par = Application.par
init = Application.init
#disp = Application.disp
simu = Application.simu
show = Application.show

model_get = Application.model_get
resetPen = Application.resetPen
process_diagram = Application.process_diagram
#describe_general = Application.describe_general
#describe_parts = Application.describe_parts
describe_MSL = Application.describe_MSL

FMU_explore_info = Application.FMU_explore_info
system_info = Application.system_info
SDG = Application.SDG

In [ ]:
run -i BPL_TEST2_Perfusion_fmpy_explore.py

In [ ]:
%matplotlib inline
plt.rcParams['figure.figsize'] = [25/2.54, 20/2.54]

In [ ]:
#describe('process')

In [ ]:
describe('culture')

In [ ]:
process_diagram()

In [ ]:
# Process parameters used throughout
par(Y=0.5, qSmax=0.75, Ks=0.1)                                      # Culture
par(filter_eps=0.10, filter_alpha_X=0.02, filter_alpha_S=0.10)      # Filter
par(S_in=30.0)                                                      # Inlet substrate conc
init(V_start=1.0, VX_start=1.0)                                     # Process initial conditions that are common
eps = Application.parValue['filter_eps']                            # Pump schedule parameter

In [ ]:
# Simulation of process with flow rate clot to wash-out for chemostat

init(VS_start=20)                                        # Process initial
par(pump1_t1=10, pump2_t1=10)                            # Pump schedule - recycle flow 10 times perfusion flow
par(pump1_F1=2.5*0.155, pump2_F1=2.5*0.155/eps)
par(pump1_t2=940, pump2_t2=940, pump1_t3=950, pump2_t3=950, pump1_t4=960, pump2_t4=960)

newplot(title='Perfusion cultivation - flow rate corresponding to maximal rate for chemostat')
simu(60)

In [ ]:
# Simulation of process with flow rate close to wash-out for chemostat

init(VS_start=20)                                        # Process initial
par(pump1_t1=10, pump2_t1=10)                            # Pump schedule - recycle flow 10 times perfusion flow
par(pump1_F1=2.5*0.155, pump2_F1=2.5*0.155/eps)
par(pump1_t2=940, pump2_t2=940, pump1_t3=950, pump2_t3=950, pump1_t4=960, pump2_t4=960)

newplot(title='Perfusion cultivation - flow rate corresponding to maximal rate for chemostat')
simu(30)
#simu(30,'cont')

In [ ]:
# Concentration factor of the filter
c=Application.model_get('filter.retentate.c[1]')/Application.model_get('filter.inlet.c[1]')
print('Conc factor of perfusion filter =', np.round(c,3))

In [ ]:
sim_res = Application.sim_res
c_data=sim_res['filter.retentate.c[1]']/sim_res['filter.inlet.c[1]']
print('Conc factor variation', np.round(min(c_data[151:]), 3), np.round(max(c_data[151:]),3))

In [ ]:
# Simulation of process with step-wise increase of pefusion rate until wash-out.
# This means that re-circulation rate change at the same time as the perfusion rate.

init(VS_start=150)                                       # Process initial varied

par(pump1_t1=12, pump2_t1=12)                            # Pump schedule - recycle flow 10 times perfusion flow
par(pump1_F1=2.5*0.155, pump2_F1=2.5*0.155/eps)
par(pump1_t2=22, pump2_t2=22)
par(pump1_F2=2.5*0.35, pump2_F2=2.5*0.35/eps)
par(pump1_t3=32, pump2_t3=32)
par(pump1_F3=2.5*0.63, pump2_F3=2.5*0.63/eps)
par(pump1_t4=42, pump2_t4=42)
par(pump1_F4=2.5*0.83, pump2_F4=2.5*0.83/eps)

newplot(title='Perfusion cultivation - step wise increase of perfusion rate and max productivity 4x chemostat')
simu(60)

In [ ]:
# Simulation without a plot and just to check typical values at high production rate
#simu(40)
#c_data=sim_res['filter.retentate.c[1]']/sim_res['filter.inlet.c[1]']
#print('Conc factor variation', np.round(min(c_data[190:]), 3), 'to', np.round(max(c_data[190:]),3))

In [ ]:
#describe('cstrProdMax')

In [ ]:
# The maximal biomass productivity before washout is obtained aroudn 40 hours
#model_get = Application.model_get
#np.round(model_get('harvesttank.inlet.F')*model_get('harvesttank.inlet.c[1]'),1)

In [ ]:
# Thus perfusion (with this filter) brings a productivity improvement of about
#np.round(23.5/5.6,1)

In [ ]:
# Finally we check the filter flow rates at time 40 hour - note the negative sign for outlfow
#Application.model_get('filter.inlet.F')

In [ ]:
#Application.model_get('filter.filtrate.F')

In [ ]:
#Application.model_get('filter.retentate.F')

## Summary
* The perfusion filter had a concentration factor of cells around 1.08 and re-cycling flow was set to a factor 10 higher than the perfusion rate and changed when perfusion rate was change to keep the ratio factor 10.
* The first simulation showed that by cell retention using perfusion filter the process could be run at a perfusion flow rate at the maximal flow rate possible for corresponding chemostat culture and cell concetration increased steadily.
* The second simulation showed that with a proper startup cell concentration, the cell concentration remained constant when perfusion rate increased in a similar way as what we see in a chemostat.
* The second simulation also showed that biomass productivity in this case was increased by a factor 4.2 compared to chemostat.
* If the perfusion rate increased to higher levels washout started but the decrase of cell concentration was slow.

Some of you who read this may have your perfusion experience with CHO-cultures. For such cultures the cell concentration do increase with increase of perfusion rate and there are understood reasons for that. But for this simplifed process as well as microbial processes they typically keep cell concentration constant when flow rate is chaged, and that under quite wide conditions. I will try come back to this phenomena in a later notebook.

In [ ]:
# List of components in the process setup and also a couple of other things like liquidphase and D
#describe('parts')

In [ ]:
describe('MSL')

In [ ]:
system_info()

In [ ]:
!lsb_release -d

In [ ]:
SDG()